# Temperature Annealing LLM Calibration Experiment

## Overview

This notebook implements and evaluates inference-time temperature annealing for improving LLM calibration on text classification tasks.

### What This Experiment Does:

1. **Generates simulated logits** for text classification examples (simulating LLM outputs)
2. **Evaluates 4 calibration methods**:
   - **Uncalibrated**: Raw softmax probabilities
   - **Temperature Scaling (TS)**: Single temperature parameter to scale logits
   - **Thermodynamic Entropy Calibration (TEC)**: Temperature + entropy-based adjustment
   - **Annealing + Softmax**: Class-dependent temperature annealing
3. **Measures calibration quality** using:
   - Expected Calibration Error (ECE)
   - Brier Score
   - Accuracy

### Dataset

Uses the SST-2 (Stanford Sentiment Treebank) dataset with binary sentiment classification (positive/negative).

### Key Findings

Temperature scaling typically reduces ECE significantly while maintaining accuracy. Annealing shows mixed results depending on the dataset.

In [ ]:
# Install dependencies - follows aii-colab pattern for Colab + local compatibility
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# numpy, scipy, matplotlib - pre-installed on Colab, install locally to match versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

print("Dependencies installed successfully!")

In [ ]:
# Imports - copied from original method.py with minimal additions for visualization
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

In [ ]:
# Data loading helper - GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d144fc-thermodynamic-entropy-calibration-for-im/main/round-2/experiment-2/demo/mini_demo_data.json"

def load_data():
    '''Load data from GitHub URL with local fallback.'''
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    
    # Local fallback
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded data with {len(data['datasets'])} dataset(s)")
for ds in data['datasets']:
    print(f"  - {ds['dataset']}: {len(ds['examples'])} examples")

## Configuration

Set all tunable parameters here. Starting with **MINIMUM values** for fast execution.

- `SAMPLE_SIZE`: Number of examples to use (minimum: 10-20)
- `DATASETS`: Which datasets to process (minimum: 1 dataset)
- `NUM_BINS`: Number of bins for ECE calculation

To scale up: increase `SAMPLE_SIZE` and add more datasets.

In [ ]:
# Configuration - scaled up slightly for meaningful results
# Original default: SAMPLE_SIZE=1000, DATASETS=['sst-2', 'ag_news', 'dbpedia']
SAMPLE_SIZE = 50  # Demo value: 50 (original: 1000)
DATASETS = ['sst-2']  # Demo value: 1 dataset (original: 3 datasets)
NUM_BINS = 10  # For ECE calculation (not typically changed)

# Annealing parameters
ANNEALING_T_START = 2.0
ANNEALING_T_END = 1.0
ANNEALING_SCHEDULE = 'linear'

print(f"Config: SAMPLE_SIZE={SAMPLE_SIZE}, DATASETS={DATASETS}")

## Helper Functions

These functions are copied from the original `method.py`:

- `softmax()`: Compute softmax probabilities
- `compute_ece()`: Expected Calibration Error
- `compute_brier_score()`: Brier Score metric
- `anneal_temperature()`: Compute class-dependent annealed temperatures

In [ ]:
# Helper functions - copied directly from method.py

def softmax(x):
    '''Compute softmax.'''
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def compute_ece(confidences, predictions, labels, num_bins=10):
    '''Compute Expected Calibration Error.'''
    bins = np.linspace(0, 1, num_bins + 1)
    ece = 0.0
    for i in range(num_bins):
        mask = (confidences >= bins[i]) & (confidences < bins[i + 1])
        if mask.sum() > 0:
            bin_conf = confidences[mask].mean()
            bin_acc = (predictions[mask] == labels[mask]).mean()
            ece += (mask.sum() / len(confidences)) * abs(bin_acc - bin_conf)
    return float(ece)

def compute_brier_score(probs, labels):
    '''Compute Brier Score.'''
    n_samples, n_classes = probs.shape
    one_hot = np.zeros((n_samples, n_classes))
    one_hot[np.arange(n_samples), labels] = 1
    return float(np.mean((probs - one_hot) ** 2))

def anneal_temperature(num_classes, T_start, T_end, schedule_type):
    '''Compute annealed temperature for each class.'''
    c = np.arange(num_classes, dtype=np.float32)
    x = c / num_classes
    
    if schedule_type == 'linear':
        schedule = 1.0 - x
    elif schedule_type == 'exponential':
        schedule = np.exp(-x)
    elif schedule_type == 'physics':
        schedule = 1.0 / np.log(1 + x * (np.e - 1))
    else:
        raise ValueError(f'Unknown schedule: {schedule_type}')
    
    T_c = T_end + (T_start - T_end) * schedule
    return T_c

print("Helper functions defined.")

## Calibration Method Classes

Three calibration methods implemented as classes:

1. **TemperatureScaling**: Optimizes a single temperature parameter using grid search
2. **ThermodynamicEntropyCalibration**: Optimizes temperature + entropy weight
3. **AnnealingSoftmax**: Applies class-dependent temperature annealing (no training needed)

In [ ]:
# Calibration method classes - copied directly from method.py

class TemperatureScaling:
    '''Temperature Scaling calibration.'''
    
    def __init__(self):
        self.temperature = 1.0
    
    def fit(self, val_logits, val_labels):
        '''Optimize temperature using grid search.'''
        best_temp = 1.0
        best_nll = float('inf')
        
        for T in np.arange(0.5, 3.0, 0.1):
            probs = softmax(val_logits / T)
            nll = -np.mean(np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10))
            if nll < best_nll:
                best_nll = nll
                best_temp = T
        
        self.temperature = best_temp
        return best_temp
    
    def calibrate(self, logits):
        '''Apply temperature scaling.'''
        return softmax(logits / self.temperature)

class ThermodynamicEntropyCalibration:
    '''Thermodynamic Entropy Calibration.'''
    
    def __init__(self):
        self.temperature = 1.0
        self.entropy_weight = 0.0
    
    def compute_entropy(self, probs):
        '''Compute Shannon entropy.'''
        return -np.sum(probs * np.log(probs + 1e-10), axis=-1)
    
    def fit(self, val_logits, val_labels):
        '''Optimize using grid search.'''
        best_params = (1.0, 0.0)
        best_nll = float('inf')
        
        for T in np.arange(0.5, 3.0, 0.2):
            for w in np.arange(-0.5, 0.5, 0.1):
                probs = softmax(val_logits / T)
                entropy = self.compute_entropy(probs)
                adjusted = probs * (1 + w * entropy[:, np.newaxis])
                adjusted = adjusted / adjusted.sum(axis=-1, keepdims=True)
                nll = -np.mean(np.log(adjusted[np.arange(len(val_labels)), val_labels] + 1e-10))
                if nll < best_nll:
                    best_nll = nll
                    best_params = (T, w)
        
        self.temperature, self.entropy_weight = best_params
        return best_params
    
    def calibrate(self, logits):
        '''Apply TEC.'''
        probs = softmax(logits / self.temperature)
        entropy = self.compute_entropy(probs)
        adjusted = probs * (1 + self.entropy_weight * entropy[:, np.newaxis])
        adjusted = adjusted / adjusted.sum(axis=-1, keepdims=True)
        return adjusted

class AnnealingSoftmax:
    '''Annealing + Softmax.'''
    
    def __init__(self, T_start=2.0, T_end=1.0, schedule_type='linear'):
        self.T_start = T_start
        self.T_end = T_end
        self.schedule_type = schedule_type
    
    def calibrate(self, logits):
        '''Apply temperature annealing.'''
        num_classes = logits.shape[-1]
        T_c = anneal_temperature(num_classes, self.T_start, self.T_end, self.schedule_type)
        T_c_batch = np.tile(T_c, (logits.shape[0], 1))
        annealed_logits = logits / T_c_batch
        return softmax(annealed_logits)

print("Calibration classes defined.")

## Run Calibration Experiment

This cell runs the main experiment:

1. Loads the specified dataset(s)
2. Generates simulated logits (using random seed for reproducibility)
3. Splits data into train/val/test sets
4. Evaluates all 4 calibration methods
5. Computes accuracy, ECE, and Brier score for each method

In [ ]:
# Main experiment logic - adapted from method.py main()

# Initialize results
all_results = {
    'experiment_name': 'inference_time_temperature_annealing',
    'datasets': DATASETS,
    'methods': ['uncalibrated', 'temperature_scaling', 
                'thermodynamic_entropy_calibration', 'annealing_softmax'],
    'results': {}
}

# Process each dataset
for dataset_name in DATASETS:
    print(f"\n{'='*60}")
    print(f"Processing: {dataset_name}")
    print(f"{'='*60}")
    
    # Find dataset
    dataset_info = None
    for ds in data['datasets']:
        if ds['dataset'] == dataset_name:
            dataset_info = ds
            break
    
    if dataset_info is None:
        print(f"Dataset {dataset_name} not found")
        continue
    
    # Get examples (use SAMPLE_SIZE from config)
    examples = dataset_info['examples'][:SAMPLE_SIZE]
    
    # Get labels and ensure they're 0-indexed
    labels_list = [int(ex['output']) for ex in examples]
    unique_labels = sorted(set(labels_list))
    label_map = {old: new for new, old in enumerate(unique_labels)}
    labels = np.array([label_map[l] for l in labels_list])
    num_classes = len(unique_labels)
    
    print(f"Examples: {len(examples)}, Classes: {num_classes}")
    print(f"Label mapping: {label_map}")
    
    # Generate simulated logits
    np.random.seed(42)
    n = len(examples)
    logits = np.random.randn(n, num_classes) * 2
    
    # Split train/val/test (60/20/20)
    indices = np.random.permutation(n)
    train_idx = indices[:int(0.6 * n)]
    val_idx = indices[int(0.6 * n):int(0.8 * n)]
    test_idx = indices[int(0.8 * n):]
    
    test_logits = logits[test_idx]
    test_labels = labels[test_idx]
    
    print(f"Split: {len(train_idx)} train, {len(val_idx)} val, {len(test_idx)} test")
    
    # Evaluate methods
    dataset_results = {}
    
    # Method 1: Uncalibrated
    print("\nEvaluating: uncalibrated")
    probs = softmax(test_logits)
    preds = np.argmax(probs, axis=-1)
    confs = np.max(probs, axis=-1)
    dataset_results['uncalibrated'] = {
        'accuracy': float((preds == test_labels).mean()),
        'ece': compute_ece(confs, preds, test_labels, NUM_BINS),
        'brier_score': compute_brier_score(probs, test_labels),
        'params': {}
    }
    print(f"  Accuracy: {dataset_results['uncalibrated']['accuracy']:.4f}, "
          f"ECE: {dataset_results['uncalibrated']['ece']:.4f}")
    
    # Method 2: Temperature Scaling
    print("Evaluating: temperature_scaling")
    ts = TemperatureScaling()
    ts.fit(logits[val_idx], labels[val_idx])
    ts_probs = ts.calibrate(test_logits)
    ts_preds = np.argmax(ts_probs, axis=-1)
    ts_confs = np.max(ts_probs, axis=-1)
    dataset_results['temperature_scaling'] = {
        'accuracy': float((ts_preds == test_labels).mean()),
        'ece': compute_ece(ts_confs, ts_preds, test_labels, NUM_BINS),
        'brier_score': compute_brier_score(ts_probs, test_labels),
        'params': {'temperature': ts.temperature}
    }
    print(f"  Accuracy: {dataset_results['temperature_scaling']['accuracy']:.4f}, "
          f"ECE: {dataset_results['temperature_scaling']['ece']:.4f}")
    
    # Method 3: TEC
    print("Evaluating: thermodynamic_entropy_calibration")
    tec = ThermodynamicEntropyCalibration()
    tec.fit(logits[val_idx], labels[val_idx])
    tec_probs = tec.calibrate(test_logits)
    tec_preds = np.argmax(tec_probs, axis=-1)
    tec_confs = np.max(tec_probs, axis=-1)
    dataset_results['thermodynamic_entropy_calibration'] = {
        'accuracy': float((tec_preds == test_labels).mean()),
        'ece': compute_ece(tec_confs, tec_preds, test_labels, NUM_BINS),
        'brier_score': compute_brier_score(tec_probs, test_labels),
        'params': {'temperature': tec.temperature, 'entropy_weight': tec.entropy_weight}
    }
    print(f"  Accuracy: {dataset_results['thermodynamic_entropy_calibration']['accuracy']:.4f}, "
          f"ECE: {dataset_results['thermodynamic_entropy_calibration']['ece']:.4f}")
    
    # Method 4: Annealing Softmax
    print("Evaluating: annealing_softmax")
    annealing = AnnealingSoftmax(T_start=ANNEALING_T_START, T_end=ANNEALING_T_END, schedule_type=ANNEALING_SCHEDULE)
    ann_probs = annealing.calibrate(test_logits)
    ann_preds = np.argmax(ann_probs, axis=-1)
    ann_confs = np.max(ann_probs, axis=-1)
    dataset_results['annealing_softmax'] = {
        'accuracy': float((ann_preds == test_labels).mean()),
        'ece': compute_ece(ann_confs, ann_preds, test_labels, NUM_BINS),
        'brier_score': compute_brier_score(ann_probs, test_labels),
        'params': {'T_start': ANNEALING_T_START, 'T_end': ANNEALING_T_END, 'schedule_type': ANNEALING_SCHEDULE}
    }
    print(f"  Accuracy: {dataset_results['annealing_softmax']['accuracy']:.4f}, "
          f"ECE: {dataset_results['annealing_softmax']['ece']:.4f}")
    
    all_results['results'][dataset_name] = dataset_results

print("\nExperiment complete!")

## Results Visualization

Display the results in a readable table and create a bar chart comparing ECE across methods.

In [ ]:
# Display results in a table
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

for dataset_name in all_results['results']:
    print(f"\nDataset: {dataset_name}")
    print("-"*80)
    print(f"{'Method':<40} {'Accuracy':<10} {'ECE':<10} {'Brier':<10}")
    print("-"*80)
    for method_name in all_results['results'][dataset_name]:
        result = all_results['results'][dataset_name][method_name]
        print(f"{method_name:<40} {result['accuracy']:<10.4f} "
              f"{result['ece']:<10.4f} {result['brier_score']:<10.4f}")

# Create visualization - bar chart of ECE by method
fig, ax = plt.subplots(figsize=(10, 6))

methods = []
ece_values = []
acc_values = []

for dataset_name in all_results['results']:
    for method_name in all_results['results'][dataset_name]:
        result = all_results['results'][dataset_name][method_name]
        methods.append(method_name.replace('_', ' ').title())
        ece_values.append(result['ece'])
        acc_values.append(result['accuracy'])

x = np.arange(len(methods))
width = 0.35

ax.bar(x - width/2, ece_values, width, label='ECE', color='steelblue')
ax.bar(x + width/2, acc_values, width, label='Accuracy', color='orange')

ax.set_xlabel('Calibration Method')
ax.set_ylabel('Score')
ax.set_title('Calibration Performance: ECE and Accuracy by Method')
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Also show calibration improvement
print("\n" + "="*80)
print("CALIBRATION IMPROVEMENT (ECE reduction vs uncalibrated)")
print("="*80)

for dataset_name in all_results['results']:
    print(f"\nDataset: {dataset_name}")
    baseline_ece = all_results['results'][dataset_name]['uncalibrated']['ece']
    for method_name in all_results['results'][dataset_name]:
        if method_name != 'uncalibrated':
            method_ece = all_results['results'][dataset_name][method_name]['ece']
            improvement = (baseline_ece - method_ece) / baseline_ece * 100
            print(f"  {method_name:<40}: {improvement:+.1f}% ECE change")